# 🧪 اضغطها | Media Lite — Encoder Benchmark v1.1

نوت بوك منفصلة **للبنش مارك فقط**. لا تغيّر محرك الضغط الأساسي، ولا تعدّل قناة Telegram، ولا ترفع نتائج الاختبار إليها.

الإصدار ده يشخّص الـGPU خطوة بخطوة بدل ما يعتبر وجود T4 كفاية: FFmpeg NVENC، NVDEC على نفس الفيديو، Full GPU pipeline، وNVEncC بشكل مستقل.

كل مرشح يعمل **Warm-up ثم عدة قياسات**، والنتيجة تعتمد على الـMedian، مع قياس CPU/GPU/NVENC utilization وتأكيد أفضل النتائج على مقطع أطول.

**الهاردوير يظهر تلقائيًا:** اسم المعالج، physical cores، logical threads، الرام، اسم الـGPU وVRAM والـdriver.

> شغّل **شامل** أولًا. لو عندك T4، أي مسار GPU يفشل هيظهر FAILED/SKIPPED ومعاه السبب بدل ما يختفي من الجدول.


In [ ]:
#@title 🧪 تشغيل البنش مارك
#@markdown الافتراضي يأخذ أحدث فيديو أصلي من قناة اضغطها. يمكن تحديد Message ID أو ملف من Drive بدلًا منه.
المصدر = "أحدث فيديو أصلي من قناة اضغطها" #@param ["أحدث فيديو أصلي من قناة اضغطها", "ملف من Google Drive"]
معرف_رسالة_Telegram = "" #@param {type:"string"}
مسار_ملف_Drive = "" #@param {type:"string"}
عمق_الاختبار = "شامل" #@param ["سريع", "شامل", "أقصى"]
الحجم_المستهدف_MB = 80 #@param {type:"integer"}
اختبار_NVEncC = True #@param {type:"boolean"}
التكرارات = 3 #@param {type:"integer"}
الاحتفاظ_بملفات_الاختبار = False #@param {type:"boolean"}

import os
import re
import subprocess
import urllib.request
import urllib.error
import json
import time

REPO = "abdullahsamirashour/gpt"
BRANCH = "main"
ENGINE_REL = "telegram-smart-compressor/benchmark_engine.py"
ENGINE_PATH = "/content/media_lite_benchmark_engine.py"

SOURCE_MAP = {
    "أحدث فيديو أصلي من قناة اضغطها": "telegram",
    "ملف من Google Drive": "drive",
}
MODE_MAP = {"سريع": "quick", "شامل": "full", "أقصى": "max"}

os.environ["BENCH_SOURCE_MODE"] = SOURCE_MAP.get(المصدر, "telegram")
os.environ["BENCH_TELEGRAM_MESSAGE_ID"] = str(معرف_رسالة_Telegram or "").strip()
os.environ["BENCH_DRIVE_FILE"] = str(مسار_ملف_Drive or "").strip()
os.environ["BENCH_MODE"] = MODE_MAP.get(عمق_الاختبار, "full")
os.environ["BENCH_TARGET_MB"] = str(int(الحجم_المستهدف_MB or 80))
os.environ["BENCH_INSTALL_NVENCC"] = "1" if اختبار_NVEncC else "0"
os.environ["BENCH_REPEATS"] = str(max(1, min(5, int(التكرارات or 3))))
os.environ["BENCH_KEEP_OUTPUTS"] = "1" if الاحتفاظ_بملفات_الاختبار else "0"

def latest_sha():
    p = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{REPO}.git", f"refs/heads/{BRANCH}"],
        capture_output=True,
        text=True,
        timeout=30,
    )
    if p.returncode == 0 and p.stdout.strip():
        sha = p.stdout.strip().split()[0]
        if re.fullmatch(r"[0-9a-f]{40}", sha):
            return sha
    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPO}/git/ref/heads/{BRANCH}?t={int(time.time())}",
        headers={
            "Accept": "application/vnd.github+json",
            "User-Agent": "Media-Lite-Benchmark-Colab",
            "Cache-Control": "no-cache",
        },
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))["object"]["sha"]

def download_engine(sha):
    url = f"https://raw.githubusercontent.com/{REPO}/{sha}/{ENGINE_REL}"
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Media-Lite-Benchmark-Colab", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return response.read().decode("utf-8")

try:
    print("🧪 تحميل أحدث محرك Benchmark من GitHub...")
    sha = latest_sha()
    os.environ["BENCH_ENGINE_SHA"] = sha
    engine = download_engine(sha)
    if len(engine) < 10000:
        raise RuntimeError("benchmark-engine-too-small")
    compiled = compile(engine, ENGINE_PATH, "exec")
    print(f"✅ Benchmark engine — commit {sha[:10]}")
    exec(compiled, globals(), globals())
    run_benchmark()
except urllib.error.HTTPError as exc:
    print(f"❌ BENCH-E001 — GitHub HTTP {exc.code}")
except urllib.error.URLError:
    print("❌ BENCH-E001 — تعذر الاتصال بـGitHub.")
except SyntaxError as exc:
    print(f"❌ BENCH-E002 — Syntax error: {exc}")
except Exception as exc:
    print(f"❌ BENCH-E003 — {type(exc).__name__}: {str(exc)[:300]}")
